# 01 | Opiniones en comentarios de YouTube
## Laboratorio de Procesamiento de Lenguaje Natural
**Archivo:** `comments.csv` · **Modalidad:** individual o en parejas · **Entorno:** CPU.

**Pregunta guía:** ¿qué lenguaje aparece en los comentarios y qué tan bien puede un modelo reproducir sus etiquetas de sentimiento?

Recorrido: cargar y auditar → limpiar sin destruir el original → tokenizar → explorar vocabulario → representar con TF-IDF → clasificar → interpretar errores → generar un informe PDF.

**Cómo comenzar:** suba este cuaderno a Google Colab, ejecute las celdas numeradas en orden y seleccione `comments.csv` cuando se solicite. No necesita el archivo de videos. El código está visible y el cuaderno se entrega sin ejecutar ni salidas guardadas.

**Límites de interpretación:** se analiza una muestra, no toda la audiencia. El CSV no incluye un diccionario ni el procedimiento de anotación: se usa como **convención de trabajo revisable** `0=negativo`, `1=neutral`, `2=positivo`. Estas etiquetas no constituyen una verificación humana independiente. No hay identificación de personas ni diagnóstico de sus emociones.

## Celda 01. Preparar el entorno

Instala solo lo que falte y muestra las versiones para reproducir la práctica.

In [ ]:
# Instala únicamente bibliotecas ausentes. No actualiza las que ya funcionan.
import importlib, subprocess, sys
DEPENDENCIAS = {
    "numpy": "numpy>=1.24,<3", "pandas": "pandas>=2,<3",
    "matplotlib": "matplotlib>=3.7,<4", "sklearn": "scikit-learn>=1.3,<2",
    "scipy": "scipy>=1.10,<2", "reportlab": "reportlab>=4,<5",
    "nltk": "nltk>=3.8,<4",
}
for modulo, paquete in DEPENDENCIAS.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import re, html, json, hashlib, zipfile, shutil, platform, unicodedata
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from IPython.display import display, FileLink
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

VERSIONES = {m: getattr(importlib.import_module(m), "__version__", "disponible")
             for m in DEPENDENCIAS}
print("Entorno preparado. No se necesita GPU ni una clave de API.")
display(pd.DataFrame(VERSIONES.items(), columns=["Biblioteca", "Versión"]))

## Celda 02. Identificación y parámetros

Modifique su nombre y grupo. Las respuestas del final también aparecerán en su PDF.

In [ ]:
ESTUDIANTE = "Por completar"
GRUPO = "Por completar"
SEMILLA = 42
TOP = 10
MAX_CARACTERISTICAS = 20000
DEMOSTRAR_LEMAS = True       # Opcional: requiere acceso a WordNet en internet.
DESCARGAR_AL_FINAL = True
ETIQUETAS = {0: "Negativo", 1: "Neutral", 2: "Positivo"}
SALIDA = Path("salida_comentarios")
GRAFICOS = SALIDA / "graficos"
GRAFICOS.mkdir(parents=True, exist_ok=True)
FECHA_ANALISIS = datetime.now(ZoneInfo("America/Bogota")).strftime("%Y-%m-%d %H:%M %Z")
PARAMETROS = {"semilla": SEMILLA, "top": TOP, "max_caracteristicas": MAX_CARACTERISTICAS,
              "etiquetas": ETIQUETAS, "division": "primera partición de StratifiedGroupKFold(5) por Video ID"}
print("Carpeta de resultados:", SALIDA)

## Celda 03. Funciones de lectura, texto y gráficos

Estas funciones se reutilizan. Observe que se preservan las negaciones y que las barras comienzan en cero.

In [ ]:
def numero(valor, decimales=0):
    """Formato numérico colombiano para tablas, gráficos e informe."""
    if pd.isna(valor):
        return "N/D"
    return f"{valor:,.{decimales}f}".replace(",", "§").replace(".", ",").replace("§", ".")

def cargar_csv(nombre, requeridas):
    """Lee un archivo local o solicita su carga en Google Colab."""
    ruta = Path(nombre)
    if not ruta.is_file():
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(f"Coloque {nombre} junto al cuaderno.") from exc
        print(f"Seleccione {nombre}, no el otro CSV.")
        subidos = files.upload()
        candidatos = [Path(x) for x in subidos if x.lower().endswith(".csv")]
        if not ruta.is_file():
            if len(candidatos) != 1:
                raise ValueError("Cargue únicamente el CSV solicitado y repita esta celda.")
            ruta = candidatos[0]
    tabla = pd.read_csv(ruta, encoding="utf-8-sig")
    tabla.columns = tabla.columns.astype(str).str.strip()
    faltan = set(requeridas) - set(tabla.columns)
    if faltan:
        raise ValueError(f"El archivo no tiene estas columnas: {sorted(faltan)}")
    if tabla.empty:
        raise ValueError("El CSV está vacío.")
    return tabla, ruta

def eje(titulo, subtitulo, alto=4.6):
    """Una figura por gráfico, sin 3D ni ejes dobles."""
    fig = plt.figure(figsize=(10.8, alto))
    ax = fig.add_subplot(111)
    fig.suptitle(titulo, x=0.02, ha="left", fontsize=16, fontweight="bold", y=0.98)
    fig.text(0.02, 0.88, subtitulo, ha="left", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="both", labelsize=10)
    ax.set_axisbelow(True)
    return fig, ax

def guardar_grafico(fig, nombre):
    fig.tight_layout(rect=(0, 0.03, 1, 0.83))
    ruta = GRAFICOS / (nombre + ".png")
    fig.savefig(ruta, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return ruta

def barras(etiquetas, valores, titulo, subtitulo, nombre, unidad="", decimales=0,
           notas=None, limite=None):
    # El orden de entrada es el orden que ve el lector: primero el más relevante.
    fig, ax = eje(titulo, subtitulo)
    y = np.arange(len(valores)); valores = np.asarray(valores, dtype=float)
    ax.barh(y, valores)
    ax.set_yticks(y, [str(x) for x in etiquetas]); ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.18)
    mayor = max(float(np.nanmax(valores)), 1e-9)
    ax.set_xlim(0, limite if limite else mayor * 1.25)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: numero(x, decimales)))
    ax.set_xlabel(unidad)
    for i, valor in enumerate(valores):
        etiqueta = numero(valor, decimales)
        if notas is not None:
            etiqueta += " | " + str(notas[i])
        ax.annotate(etiqueta, (valor, i), xytext=(5, 0), textcoords="offset points",
                    va="center", fontsize=9)
    return guardar_grafico(fig, nombre)

# Las listas de palabras vacías se usan para explorar vocabulario, no para borrar
# automáticamente palabras del modelo de sentimiento.
STOP_ES = set("el la los las de del un una unos unas y o que en a al con por para es son se su sus lo como este esta estos estas mi mis tu tus me te le les ya muy mas pero si hay desde sobre entre".split())
STOP = (set(ENGLISH_STOP_WORDS) | STOP_ES) - {"no", "not", "nor", "never", "nunca", "sin"}
PATRON = r"(?u)\b[^\W\d_][\w']*\b"

def normalizar(texto):
    """No modifica el original; conserva acentos, negaciones y puntuación."""
    t = unicodedata.normalize("NFKC", html.unescape(str(texto))).lower()
    t = t.replace("’", "'").replace("‘", "'")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\bwon't\b", "will not", t)
    t = re.sub(r"\bcan't\b", "can not", t)
    t = re.sub(r"n't\b", " not", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenizar(texto):
    return re.findall(PATRON, texto)

def tabla_vocabulario(series, top=10):
    # Frecuencia documental: cada término cuenta máximo una vez en cada texto.
    vec = CountVectorizer(token_pattern=PATRON, lowercase=False, binary=True,
                          stop_words=sorted(STOP), min_df=2)
    matriz = vec.fit_transform(series)
    conteos = np.asarray(matriz.sum(axis=0)).ravel()
    tabla = pd.DataFrame({"Término": vec.get_feature_names_out(), "Textos": conteos})
    tabla["Porcentaje"] = 100 * tabla["Textos"] / len(series)
    return tabla.sort_values(["Textos", "Término"], ascending=[False, True]), vec, matriz

print("Funciones listas. Los originales permanecen intactos.")

## Celda 04. Cargar y explorar el CSV

Identifique la unidad de análisis: una fila representa un registro de comentario, no una persona.

In [ ]:
original, RUTA_CSV = cargar_csv("comments.csv", ["Video ID", "Comment", "Likes", "Sentiment"])
print(f"Registros originales: {numero(len(original))} | Columnas: {len(original.columns)}")
display(original.head(5))
diagnostico = pd.DataFrame({"Columna": original.columns,
    "Tipo": original.dtypes.astype(str).values,
    "Vacíos": original.isna().sum().values})
display(diagnostico)

## Celda 05. Auditar y limpiar de forma trazable

Se retiran duplicados exactos, textos vacíos y etiquetas inválidas. Los likes desconocidos permanecen vacíos, no se convierten en cero.

In [ ]:
base = original.loc[:, ~original.columns.str.match(r"^Unnamed")].copy()
duplicados_exactos = base[base.duplicated(keep="first")].copy()
df = base.drop_duplicates().copy()
df["registro_origen"] = df.index  # Número de registro; no es la línea física del CSV.
vacio = df["Comment"].isna() | df["Comment"].fillna("").astype(str).str.strip().eq("")
sin_video = df["Video ID"].isna() | df["Video ID"].fillna("").astype(str).str.strip().eq("")
sent = pd.to_numeric(df["Sentiment"], errors="coerce")
etiqueta_invalida = ~sent.isin(ETIQUETAS)
rechazados = df[vacio | sin_video | etiqueta_invalida].copy()
rechazados["motivo"] = np.select(
    [vacio.loc[rechazados.index], sin_video.loc[rechazados.index]],
    ["Texto vacío", "Video ID vacío"], default="Etiqueta inválida")
df = df.loc[~(vacio | sin_video | etiqueta_invalida)].copy()
df["Sentiment"] = pd.to_numeric(df["Sentiment"]).astype(int)
df["sentimiento"] = df["Sentiment"].map(ETIQUETAS)
df["Likes"] = pd.to_numeric(df["Likes"], errors="coerce")
likes_invalidos = int((df["Likes"] < 0).sum())
df.loc[df["Likes"] < 0, "Likes"] = np.nan
df["Comment"] = df["Comment"].astype(str)
df["Video ID"] = df["Video ID"].astype(str).str.strip()
df = df.reset_index(drop=True)
if len(df) < 50 or df["Sentiment"].nunique() != 3:
    raise ValueError("Se necesitan al menos 50 comentarios y las tres clases para esta práctica.")
auditoria = pd.DataFrame([
    ("Registros originales", len(original)), ("Duplicados exactos retirados", len(duplicados_exactos)),
    ("Registros rechazados después de deduplicar", len(rechazados)),
    ("Likes negativos convertidos en vacío", likes_invalidos),
    ("Comentarios conservados", len(df)), ("Videos representados", df["Video ID"].nunique())
], columns=["Control", "Cantidad"])
display(auditoria)
assert len(original) == len(duplicados_exactos) + len(rechazados) + len(df)

## Celda 06. Normalizar y tokenizar

Compare original, texto normalizado y tokens. Los emojis se conservan en el texto, aunque este modelo de palabras no los representa.

In [ ]:
df["texto_normalizado"] = df["Comment"].map(normalizar)
df["tokens"] = df["texto_normalizado"].map(tokenizar)
df["n_palabras"] = df["tokens"].map(len)
df["n_caracteres"] = df["Comment"].str.len()
df["tiene_pregunta"] = df["Comment"].str.contains("?", regex=False)
df["tiene_exclamacion"] = df["Comment"].str.contains("!", regex=False)
df["tiene_negacion"] = df["tokens"].map(lambda t: bool(set(t) & {"no", "not", "never", "nor", "nunca", "sin"}))
display(df[["Comment", "texto_normalizado", "tokens", "n_palabras"]].head(4))
print("Ejemplo de negación:", normalizar("I don't like this video!"))
print("Sin palabras detectables:", int(df["n_palabras"].eq(0).sum()))
print("La tokenización es aproximada: no equivale a entender el significado.")

## Celda 07. Lematización: demostración controlada

Observe cómo cambia una palabra según su categoría gramatical. WordNet es para inglés; no se aplica automáticamente a otros idiomas.

In [ ]:
# Demostración opcional: WordNet necesita descargar un recurso la primera vez.
# No se aplica indiscriminadamente a todo el corpus, que puede ser multilingüe.
# pos='n': sustantivo; pos='v': verbo. El contexto gramatical cambia el resultado.
estado_lemas = "Demostración desactivada; no se aplicó al modelo."
if DEMOSTRAR_LEMAS:
    import nltk
    from nltk.stem import WordNetLemmatizer
    try:
        lematizador = WordNetLemmatizer()
        try:
            lematizador.lemmatize("videos", pos="n")
        except LookupError:
            if not nltk.download("wordnet", quiet=True, raise_on_error=False):
                raise RuntimeError("No fue posible descargar WordNet.")
        ejemplos_lemas = [("videos", "n"), ("comments", "n"), ("running", "v"),
                          ("liked", "v"), ("better", "a")]
        tabla_lemas = pd.DataFrame([
            {"Palabra": w, "Categoría gramatical": p,
             "Lema": lematizador.lemmatize(w, pos=p)} for w, p in ejemplos_lemas])
        display(tabla_lemas)
        estado_lemas = "Demostración de WordNet realizada; no se aplicó al modelo."
    except (LookupError, RuntimeError, OSError) as exc:
        estado_lemas = "WordNet no disponible; se continuó sin lematización."
        print(estado_lemas, str(exc))
        print("El análisis y el PDF funcionan sin este recurso opcional.")
else:
    print(estado_lemas)

## Celda 08. Estadística del corpus y balance de clases

Interprete mediana y percentiles además del promedio. La distribución es la de las etiquetas existentes, no una nueva predicción.

In [ ]:
estadisticas = df[["n_palabras", "n_caracteres", "Likes"]].describe(percentiles=[.25,.5,.75,.9,.95]).T
estadisticas = estadisticas.reset_index(names="Variable")
display(estadisticas.round(2))
distribucion = df["sentimiento"].value_counts().reindex(ETIQUETAS.values(), fill_value=0).rename_axis("Sentimiento").reset_index(name="Comentarios")
distribucion["Porcentaje"] = 100 * distribucion["Comentarios"] / len(df)
display(distribucion.round(2))
barras(distribucion["Sentimiento"], distribucion["Porcentaje"],
       "¿Cómo se distribuyen las etiquetas?", f"Base depurada: {numero(len(df))} comentarios | Convención 0/1/2 revisable",
       "01_sentimientos", "Porcentaje de comentarios (%)", 1, limite=100,
       notas=["n=" + numero(x) for x in distribucion["Comentarios"]])
franjas = pd.cut(df["n_palabras"], bins=[-1,10,30,60,120,np.inf],
    labels=["0 a 10 palabras", "11 a 30", "31 a 60", "61 a 120", "Más de 120"])
longitudes = franjas.value_counts(sort=False).rename_axis("Longitud").reset_index(name="Comentarios")
longitudes["Porcentaje"] = 100 * longitudes["Comentarios"] / len(df)
barras(longitudes["Longitud"], longitudes["Porcentaje"], "¿Qué tan extensos son los comentarios?",
       f"Mediana: {numero(df.n_palabras.median())} palabras | Se incluyen todos los registros conservados",
       "02_longitudes", "Porcentaje de comentarios (%)", 1)

## Celda 09. Vocabulario y expresiones frecuentes

Se cuenta en cuántos comentarios aparece un término. Los bigramas conservan la adyacencia del texto; no se crean uniendo palabras que antes estaban separadas.

In [ ]:
vocabulario, vector_conteo, matriz_conteo = tabla_vocabulario(df["texto_normalizado"], TOP)
display(vocabulario.head(TOP).round(2))
barras(vocabulario.head(TOP)["Término"], vocabulario.head(TOP)["Porcentaje"],
       "Palabras presentes en más comentarios", "Frecuencia documental; sin palabras vacías inglesas/españolas | No mide importancia causal",
       "03_vocabulario", "Comentarios que contienen el término (%)", 1)

conteo_bigramas = Counter()
for tokens in df["tokens"]:
    pares = {a + " " + b for a, b in zip(tokens, tokens[1:])
             if a not in STOP and b not in STOP and len(a) > 1 and len(b) > 1}
    conteo_bigramas.update(pares)
bigramas = pd.DataFrame(conteo_bigramas.most_common(), columns=["Bigrama", "Comentarios"])
bigramas["Porcentaje"] = 100 * bigramas["Comentarios"] / len(df)
display(bigramas.head(TOP).round(2))
barras(bigramas.head(TOP)["Bigrama"], bigramas.head(TOP)["Porcentaje"],
       "Expresiones de dos palabras más extendidas", "Pares adyacentes de palabras de contenido; se preservan negaciones",
       "04_bigramas", "Comentarios que contienen la expresión (%)", 1)

## Celda 10. Manipular y consultar los comentarios

Cambie FILTRO_TEXTO, FILTRO_SENTIMIENTO y MIN_LIKES. La búsqueda es literal y no modifica el corpus ni las estadísticas generales.

In [ ]:
FILTRO_TEXTO = "apple"
FILTRO_SENTIMIENTO = "Todos"   # Todos, Negativo, Neutral o Positivo.
MIN_LIKES = 0
mask = df["texto_normalizado"].str.contains(normalizar(FILTRO_TEXTO), regex=False, na=False)
if FILTRO_SENTIMIENTO != "Todos":
    if FILTRO_SENTIMIENTO not in ETIQUETAS.values():
        raise ValueError("Sentimiento no reconocido.")
    mask &= df["sentimiento"].eq(FILTRO_SENTIMIENTO)
if MIN_LIKES > 0:
    mask &= df["Likes"].ge(MIN_LIKES)
consulta = df.loc[mask, ["Video ID", "Comment", "sentimiento", "Likes"]].sort_values("Likes", ascending=False)
print("Coincidencias:", numero(len(consulta)))
display(consulta.head(10))
# Se exporta la consulta completa al final. Un cero aquí puede significar que no hay coincidencias.

## Celda 11. Separar entrenamiento y prueba sin mezclar videos

Primero se aparta una prueba por Video ID. Los textos idénticos con etiquetas contradictorias se excluyen solo del modelado; los repetidos se conservan una vez. TF-IDF se ajustará únicamente con entrenamiento.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
candidatos = df.loc[df["n_palabras"] > 0].copy()
etiquetas_por_texto = candidatos.groupby("texto_normalizado")["Sentiment"].nunique()
textos_conflictivos = etiquetas_por_texto[etiquetas_por_texto > 1].index
conflictivos = candidatos[candidatos["texto_normalizado"].isin(textos_conflictivos)].copy()
base_modelo = candidatos[~candidatos["texto_normalizado"].isin(textos_conflictivos)].copy()
repetidos_modelo = int(base_modelo.duplicated("texto_normalizado").sum())
base_modelo = base_modelo.drop_duplicates("texto_normalizado").reset_index(drop=True)
if base_modelo["Video ID"].nunique() < 5:
    raise ValueError("Se requieren al menos cinco videos para separar grupos.")
partidor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
idx_entrena, idx_prueba = next(partidor.split(base_modelo["Comment"], base_modelo["Sentiment"], groups=base_modelo["Video ID"]))
entrena = base_modelo.iloc[idx_entrena].copy()
prueba = base_modelo.iloc[idx_prueba].copy()
assert set(entrena["Video ID"]).isdisjoint(set(prueba["Video ID"]))
assert set(entrena["texto_normalizado"]).isdisjoint(set(prueba["texto_normalizado"]))
if set(entrena["Sentiment"]) != set(ETIQUETAS) or set(prueba["Sentiment"]) != set(ETIQUETAS):
    raise ValueError("La partición no contiene las tres clases. Revise la distribución por video.")
particiones = pd.DataFrame([
    ("Entrenamiento",len(entrena),entrena["Video ID"].nunique()),
    ("Prueba",len(prueba),prueba["Video ID"].nunique())], columns=["Partición","Comentarios","Videos"])
display(particiones)
print("Registros de textos contradictorios excluidos del modelo:", len(conflictivos))
print("Repeticiones adicionales excluidas del modelo:", repetidos_modelo)
print("La prueba permanece reservada: no es un conjunto para elegir parámetros.")

## Celda 12. Representar el lenguaje y entrenar

TF-IDF pondera palabras y bigramas. La regresión logística aprende las etiquetas. El modelo de referencia siempre predice la clase mayoritaria del entrenamiento.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
modelo = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=normalizar, lowercase=False,
        token_pattern=PATRON + r"|[!?]", ngram_range=(1,2), min_df=2,
        max_features=MAX_CARACTERISTICAS, sublinear_tf=True)),
    ("clasificador", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEMILLA))
])
with threadpool_limits(limits=2):
    modelo.fit(entrena["Comment"], entrena["Sentiment"])
prediccion = modelo.predict(prueba["Comment"])
base_referencia = DummyClassifier(strategy="most_frequent")
base_referencia.fit(np.zeros((len(entrena),1)), entrena["Sentiment"])
prediccion_base = base_referencia.predict(np.zeros((len(prueba),1)))

# Inspección pequeña: no convertimos toda la matriz dispersa a una matriz densa.
tfidf = modelo.named_steps["tfidf"]
ejemplo_matriz = tfidf.transform(entrena["Comment"].head(3))
indices = np.argsort(np.asarray(ejemplo_matriz.sum(axis=0)).ravel())[-10:][::-1]
ejemplo_tfidf = pd.DataFrame(ejemplo_matriz[:,indices].toarray(),
    columns=tfidf.get_feature_names_out()[indices], index=["Comentario 1","Comentario 2","Comentario 3"])
display(ejemplo_tfidf.round(3))
print("Dimensión de entrenamiento:", (len(entrena),len(tfidf.vocabulary_)))
print("El modelo utiliza texto, no Likes ni Video ID como predictores.")

## Celda 13. Evaluar con métricas y matriz de confusión

Accuracy mide acierto global; F1 macro da el mismo peso a cada clase. Recall indica qué proporción de una clase se detecta. Los resultados se calculan solo en prueba.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay)
y_prueba = prueba["Sentiment"]
metricas = pd.DataFrame([
    {"Modelo": nombre, "Accuracy": accuracy_score(y_prueba, pred),
     "Accuracy balanceada": balanced_accuracy_score(y_prueba,pred),
     "F1 macro": f1_score(y_prueba,pred,average="macro",zero_division=0)}
    for nombre,pred in [("Referencia mayoritaria",prediccion_base),("TF-IDF + logística",prediccion)]])
display(metricas.round(4))
reporte_clases = pd.DataFrame(classification_report(y_prueba,prediccion,
    labels=list(ETIQUETAS), target_names=list(ETIQUETAS.values()), output_dict=True,
    zero_division=0)).T.loc[list(ETIQUETAS.values())].reset_index(names="Clase")
display(reporte_clases.round(3))
barras(metricas["Modelo"],100*metricas["F1 macro"],"¿El modelo supera la referencia?",
       f"F1 macro en {numero(len(prueba))} comentarios de videos reservados | Más alto es mejor",
       "05_modelo", "F1 macro (escala 0 a 100)",1,limite=100)
cm = confusion_matrix(y_prueba,prediccion,labels=list(ETIQUETAS))
cm_pct = 100 * cm / cm.sum(axis=1,keepdims=True)
fig,ax = eje("¿Dónde acierta y dónde se confunde?", "Porcentaje por fila: cada sentimiento real suma 100%", alto=5.0)
ConfusionMatrixDisplay(cm_pct,display_labels=list(ETIQUETAS.values())).plot(
    ax=ax, values_format=".1f",colorbar=False)
ax.set_xlabel("Sentimiento predicho"); ax.set_ylabel("Etiqueta del archivo")
guardar_grafico(fig,"06_confusion")
print("Soporte por clase en prueba:", dict(zip(ETIQUETAS.values(),cm.sum(axis=1))))

## Celda 14. Revisar errores y probar comentarios propios

Revise casos donde el modelo y la etiqueta discrepan; cualquiera de los dos puede fallar. Los puntajes del modelo no son garantías de certeza.

In [ ]:
resultados_prueba = prueba[["registro_origen","Video ID","Comment","sentimiento"]].copy()
resultados_prueba["prediccion"] = pd.Series(prediccion,index=prueba.index).map(ETIQUETAS)
resultados_prueba["acierto"] = resultados_prueba["sentimiento"].eq(resultados_prueba["prediccion"])
errores = resultados_prueba[~resultados_prueba["acierto"]].copy()
display(errores.head(8))

MIS_COMENTARIOS = ["I love this video, it is useful!", "I do not like this video.",
                  "The video was published yesterday."]
probabilidades = modelo.predict_proba(MIS_COMENTARIOS)
pred_propias = modelo.predict(MIS_COMENTARIOS)
pruebas_propias = pd.DataFrame({"Comentario":MIS_COMENTARIOS,
    "Predicción":[ETIQUETAS[int(x)] for x in pred_propias],
    "Puntaje máximo del modelo":probabilidades.max(axis=1)})
display(pruebas_propias)
print("Se recomienda probar inglés, negación y sarcasmo; el modelo no entiende todos los contextos.")

## Celda 15. Escribir la interpretación del estudiante

Complete estas respuestas antes de generar el PDF. No cambie las conclusiones automáticas: use las respuestas para discutirlas.

In [ ]:
RESPUESTAS = {
 "¿Qué cambia al limpiar el archivo?": "Por completar.",
 "¿Qué palabra o expresión le llamó la atención y por qué?": "Por completar.",
 "¿Por qué no basta con mirar accuracy?": "Por completar.",
 "¿Qué error del modelo revisó y cómo lo explica?": "Por completar.",
 "¿Qué mejora propondría y cómo la validaría sin usar la prueba?": "Por completar."
}
mejor_clase = distribucion.loc[distribucion["Comentarios"].idxmax()]
f1 = float(metricas.loc[metricas["Modelo"].eq("TF-IDF + logística"),"F1 macro"].iloc[0])
f1_base = float(metricas.loc[metricas["Modelo"].eq("Referencia mayoritaria"),"F1 macro"].iloc[0])
delta = 100 * (f1 - f1_base)
clase_dificil = reporte_clases.sort_values("recall").iloc[0]
HALLAZGOS = [
 f"Se conservaron {numero(len(df))} de {numero(len(original))} registros; {numero(len(duplicados_exactos))} duplicados exactos se retiraron.",
 f"La etiqueta más frecuente es {mejor_clase['Sentimiento']}: {numero(mejor_clase['Porcentaje'],1)}% del corpus depurado. No equivale a la opinión de toda la audiencia.",
 f"La mediana de extensión es {numero(df.n_palabras.median())} palabras por comentario.",
 f"F1 macro del modelo: {numero(100*f1,1)} sobre 100; referencia: {numero(100*f1_base,1)}. Diferencia: {numero(delta,1)} puntos.",
 f"La clase con menor recall es {clase_dificil['Clase']} ({numero(100*clase_dificil['recall'],1)}%). Conviene revisar sus errores y la calidad de las etiquetas."
]
for h in HALLAZGOS:
    print(h)

## Celda 16. Preparar la plantilla PDF

El informe incluye cifras calculadas, gráficos, controles de calidad, límites y sus respuestas. No requiere servicios externos.

In [ ]:
# Estas funciones organizan el informe. No alteran los resultados del análisis.
from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image as ImagenPDF, PageBreak, KeepTogether)
from PIL import Image as ImagenPIL

ESTILOS = getSampleStyleSheet()
ESTILOS.add(ParagraphStyle(name="PortadaPLN", fontName="Helvetica-Bold",
    fontSize=26, leading=29, spaceAfter=12, textColor=colors.HexColor("#15314B")))
ESTILOS.add(ParagraphStyle(name="SeccionPLN", fontName="Helvetica-Bold",
    fontSize=15, leading=19, spaceBefore=8, spaceAfter=10,
    textColor=colors.HexColor("#15314B")))
ESTILOS.add(ParagraphStyle(name="CuerpoPLN", fontName="Helvetica",
    fontSize=10, leading=14, spaceAfter=8))
ESTILOS.add(ParagraphStyle(name="NotaPLN", fontName="Helvetica",
    fontSize=8, leading=11, spaceAfter=6))
ESTILOS.add(ParagraphStyle(name="CeldaPLN", fontName="Helvetica", fontSize=8, leading=10))
ESTILOS.add(ParagraphStyle(name="KpiPLN", fontName="Helvetica-Bold", fontSize=19,
    leading=24, spaceAfter=3, textColor=colors.HexColor("#15314B")))
ANCHO_PDF = A4[0] - 88

def texto_pdf(valor):
    # Helvetica cubre el español. Los textos originales multilingües se conservan
    # íntegros en los CSV; en el PDF se sustituyen glifos no disponibles.
    original = str(valor).replace("\x00", "")
    seguro = "".join(ch if ch.encode("cp1252", errors="ignore") else "\ufffd" for ch in original)
    return re.sub(r"\ufffd+", "[ver CSV]", seguro)

def parrafo(texto, estilo="CuerpoPLN"):
    return Paragraph(escape(texto_pdf(texto)).replace("\n", "<br/>"), ESTILOS[estilo])

def tabla_pdf(tabla, anchos=None, max_filas=15):
    t = tabla.head(max_filas).copy()
    def celda(x):
        return parrafo("N/D" if pd.isna(x) else str(x), "CeldaPLN")
    datos = [[celda(x) for x in t.columns]] + [[celda(x) for x in fila]
             for fila in t.itertuples(index=False, name=None)]
    anchos = anchos or [ANCHO_PDF / len(t.columns)] * len(t.columns)
    objeto = Table(datos, colWidths=anchos, repeatRows=1, hAlign="LEFT")
    objeto.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#EAF0F4")),
        ("LINEBELOW", (0,0), (-1,0), 0.8, colors.HexColor("#8CA0B1")),
        ("LINEBELOW", (0,1), (-1,-1), 0.3, colors.HexColor("#DEE4E9")),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6),
        ("LEFTPADDING", (0,0), (-1,-1), 7), ("RIGHTPADDING", (0,0), (-1,-1), 7)]))
    return objeto

def indicadores_pdf(items):
    datos = [[ [parrafo(valor, "KpiPLN"), parrafo(nombre, "NotaPLN")]
              for nombre, valor in items ]]
    t = Table(datos, colWidths=[ANCHO_PDF/len(items)]*len(items), hAlign="LEFT")
    t.setStyle(TableStyle([("BACKGROUND", (0,0),(-1,-1),colors.HexColor("#F2F5F7")),
      ("VALIGN", (0,0),(-1,-1),"TOP"), ("TOPPADDING", (0,0),(-1,-1),10),
      ("BOTTOMPADDING", (0,0),(-1,-1),10), ("LEFTPADDING", (0,0),(-1,-1),10)]))
    return t

def imagen_pdf(nombre, alto_max=223):
    ruta = GRAFICOS / (nombre + ".png")
    with ImagenPIL.open(ruta) as im:
        w, h = im.size
    escala = min(ANCHO_PDF/w, alto_max/h)
    return ImagenPDF(str(ruta), width=w*escala, height=h*escala, hAlign="LEFT")

def pagina_graficos(elementos, titulo, graficos, notas):
    elementos.extend([PageBreak(), parrafo(titulo, "SeccionPLN")])
    for nombre, nota in zip(graficos, notas):
        elementos.extend([imagen_pdf(nombre), Spacer(1,5), parrafo(nota, "NotaPLN"), Spacer(1,8)])

def pie_pagina(canvas, doc):
    canvas.saveState()
    canvas.setStrokeColor(colors.HexColor("#D5DFE7"))
    canvas.line(44, 42, A4[0]-44, 42)
    canvas.setFont("Helvetica", 8)
    canvas.drawString(44, 29, "PROCESAMIENTO DE LENGUAJE NATURAL | Informe reproducible")
    canvas.drawRightString(A4[0]-44, 29, f"{doc.page}")
    canvas.restoreState()

def crear_pdf(elementos, nombre):
    ruta = SALIDA/nombre
    doc = SimpleDocTemplate(str(ruta), pagesize=A4, rightMargin=44, leftMargin=44,
          topMargin=42, bottomMargin=56, title=TITULO_INFORME, author=ESTUDIANTE)
    doc.build(elementos, onFirstPage=pie_pagina, onLaterPages=pie_pagina)
    return ruta

print("Plantilla del informe preparada.")

## Celda 17. Generar el informe gerencial en PDF

El PDF se actualiza con los resultados de esta ejecución. La fecha corresponde al análisis, no a la captura de los videos.

In [ ]:
TITULO_INFORME = "Comentarios de YouTube | Análisis de PLN"
contenido = [parrafo("LABORATORIO 01 / INFORME GERENCIAL", "NotaPLN"),
    parrafo("Lenguaje y sentimientos\nen comentarios", "PortadaPLN"),
    parrafo(f"Estudiante: {ESTUDIANTE} | Grupo: {GRUPO}\nAnálisis: {FECHA_ANALISIS}","NotaPLN"),
    indicadores_pdf([("Comentarios depurados",numero(len(df))),
                     ("Videos representados",numero(df['Video ID'].nunique())),
                     ("F1 macro / 100",numero(100*f1,1))]), Spacer(1,12),
    parrafo("Resumen para la toma de decisiones", "SeccionPLN")]
for h in HALLAZGOS[:4]: contenido.append(parrafo(h))
contenido.extend([parrafo("Control de calidad", "SeccionPLN"),tabla_pdf(auditoria, [ANCHO_PDF*.75,ANCHO_PDF*.25]),
    Spacer(1,8),parrafo("Fuente: comments.csv suministrado para la clase. No se conoce el método de muestreo ni el procedimiento de etiquetado. Convención revisable: 0 negativo, 1 neutral y 2 positivo.","NotaPLN")])
pagina_graficos(contenido,"01 / Perfil del corpus",["01_sentimientos","02_longitudes"],[
    "Las barras resumen etiquetas ya existentes. No se infiere satisfacción de toda la audiencia ni se identifican personas.",
    "Los intervalos incluyen todos los comentarios depurados. La mediana describe el centro sin quedar dominada por textos extremadamente largos."])
pagina_graficos(contenido,"02 / Evidencia lingüística",["03_vocabulario","04_bigramas"],[
    "Frecuencia documental: un término cuenta una vez por comentario. Una palabra frecuente no es automáticamente la más importante para predecir.",
    "Los bigramas se forman antes de filtrar palabras vacías, manteniendo la adyacencia original. Las listas de inglés y español no cubren todos los idiomas."])
pagina_graficos(contenido,"03 / Desempeño del clasificador",["05_modelo","06_confusion"],[
    f"Prueba con {numero(len(prueba))} comentarios y {numero(prueba['Video ID'].nunique())} videos no usados para entrenar. TF-IDF se ajustó solo con entrenamiento.",
    "La diagonal representa coincidencias con la etiqueta del archivo. Cada fila suma 100%; el soporte de cada clase se presenta en la página siguiente."])
contenido.extend([PageBreak(),parrafo("04 / Validación, límites y acciones", "SeccionPLN"),
    parrafo("Desempeño por clase", "SeccionPLN")])
clases_pdf = reporte_clases.copy()
for campo in ["precision","recall","f1-score"]:
    clases_pdf[campo] = clases_pdf[campo].map(lambda x:numero(100*x,1))
clases_pdf["support"] = clases_pdf["support"].map(numero)
clases_pdf.columns=["Clase","Precisión (%)","Recall (%)","F1 / 100","Casos"]
contenido.extend([tabla_pdf(clases_pdf),Spacer(1,8),parrafo(HALLAZGOS[4]),
    parrafo(f"Base de modelado: {numero(len(base_modelo))} textos. Se apartaron {numero(len(conflictivos))} registros con texto idéntico y etiquetas contradictorias, además de {numero(repetidos_modelo)} repeticiones de texto. Estas exclusiones no alteran las estadísticas del corpus depurado."),
    parrafo("Alcance de la evaluación", "SeccionPLN"),
    parrafo("Se utilizó una única partición fija por video, no validación cruzada completa. Los resultados no tienen un intervalo de confianza calculado ni prueban generalización a toda la plataforma. Las etiquetas del CSV pueden contener errores."),
    parrafo("Acciones recomendadas", "SeccionPLN"),
    parrafo("Revisar manualmente errores de la clase con menor recall y comentarios con sarcasmo o negación. Confirmar el diccionario de etiquetas. Para comparar modelos, usar validación por grupos solo dentro del entrenamiento y reservar la prueba final."),
    parrafo("Decisiones de procesamiento", "SeccionPLN"),
    parrafo("Se conservó el texto original. El modelo usa palabras y bigramas, signos ! y ?, y no elimina negaciones. No representa emojis ni contexto audiovisual. La lematización se limita a una demostración opcional de inglés."),
    parrafo(estado_lemas,"NotaPLN")])
contenido.extend([PageBreak(),parrafo("05 / Interpretación y trazabilidad", "SeccionPLN")])
for pregunta,respuesta in RESPUESTAS.items():
    contenido.extend([parrafo(pregunta,"SeccionPLN"),parrafo(respuesta)])
contenido.extend([Spacer(1,10),parrafo("Documentación técnica", "SeccionPLN"),
    parrafo("scikit-learn: extracción de características, TfidfVectorizer, StratifiedGroupKFold y prevención de fuga de información.\nhttps://scikit-learn.org/stable/modules/feature_extraction.html\nhttps://scikit-learn.org/stable/common_pitfalls.html\nhttps://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html", "NotaPLN"),
    parrafo("NLTK: WordNetLemmatizer. https://www.nltk.org/api/nltk.stem.wordnet.html\nReportLab: Platypus. https://docs.reportlab.com/reportlab/userguide/ch5_platypus/", "NotaPLN"),
    parrafo("Los CSV de salida contienen los datos depurados, consultas, exclusiones, predicciones y métricas. metadatos.json registra parámetros, versiones y la huella SHA-256 del archivo original.","NotaPLN")])
RUTA_PDF = crear_pdf(contenido,"informe_comentarios.pdf")
print("PDF generado:", RUTA_PDF)

## Celda 18. Exportar el informe y las evidencias

Se descargan un PDF independiente y un ZIP con el PDF, gráficos PNG, tablas CSV y metadatos. Los CSV originales no se sobrescriben.

In [ ]:
# Convertimos listas de tokens a JSON para conservarlas de forma legible en CSV.
comentarios_exportar = df.copy()
comentarios_exportar["tokens"] = comentarios_exportar["tokens"].map(lambda x:json.dumps(x,ensure_ascii=False))
TABLAS_EXPORTAR = {"comentarios_limpios":comentarios_exportar,"auditoria":auditoria,
    "duplicados_exactos":duplicados_exactos,"rechazados":rechazados,
    "textos_con_etiquetas_contradictorias":conflictivos.drop(columns=["tokens"],errors="ignore"),
    "estadistica_descriptiva":estadisticas,"sentimientos":distribucion,
    "longitud_comentarios":longitudes,"vocabulario":vocabulario,"bigramas":bigramas,
    "consulta_comentarios":consulta,"particiones":particiones,"metricas_modelos":metricas,
    "metricas_por_clase":reporte_clases,"predicciones_prueba":resultados_prueba,
    "errores_modelo":errores,"pruebas_propias":pruebas_propias}

# CSV: datos y tablas para auditoría. JSON: parámetros, versiones y huella del original.
for nombre, tabla in TABLAS_EXPORTAR.items():
    tabla.to_csv(SALIDA / (nombre + ".csv"), index=False, encoding="utf-8-sig")
metadatos = {
    "estudiante": ESTUDIANTE, "grupo": GRUPO, "fecha_analisis": FECHA_ANALISIS,
    "archivo_origen": RUTA_CSV.name,
    "sha256_origen": hashlib.sha256(RUTA_CSV.read_bytes()).hexdigest(),
    "parametros": PARAMETROS, "bibliotecas": VERSIONES,
    "python": platform.python_version(), "estado_lematizacion": estado_lemas,
    "nota": "Cifras históricas del CSV. Fecha de extracción y método de muestreo no documentados."
}
(SALIDA/"metadatos.json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8")
zip_salida = Path(shutil.make_archive(str(SALIDA), "zip", root_dir=SALIDA))
print("Informe PDF:", RUTA_PDF)
print("Paquete con PDF, CSV, gráficos y metadatos:", zip_salida)
display(FileLink(str(RUTA_PDF)))
display(FileLink(str(zip_salida)))
if DESCARGAR_AL_FINAL:
    try:
        from google.colab import files
        files.download(str(RUTA_PDF))
        files.download(str(zip_salida))
    except ImportError:
        print("Fuera de Colab: abra los enlaces anteriores.")
print("Guarde también una copia de su cuaderno resuelto como evidencia.")

### Reto de cierre
Escriba tres comentarios propios que incluyan una negación, una opinión neutral y sarcasmo. Analice un error, no solo un acierto. Proponga una mejora del procesamiento y explique cómo la evaluaría usando exclusivamente el entrenamiento para elegir parámetros.

**Entregue:** su cuaderno resuelto, `informe_comentarios.pdf` y el ZIP de evidencias. Para regenerar el informe tras editar sus respuestas, vuelva a ejecutar las celdas 15 a 18.